# Entrenar un segmentador de palabras (one-shot / few-shot) — Etiquetado de Manuscritos

Este notebook implementa la receta de **De Nardin et al. (WACV 2024)** — *A One-Shot Learning Approach to Document Layout Segmentation of Ancient Arabic Manuscripts* — adaptada de "texto principal / texto marginal / fondo" a **"palabra / fondo"**, usando como ground truth las palabras ya etiquetadas manualmente en el proyecto (exportadas con `backend/scripts/export_word_masks.py`).

Idea clave del paper: con **1-3 páginas ya anotadas** (no hace falta un dataset grande) más un módulo de *Dynamic Instance Generation* (recortes en grid + recortes aleatorios como augmentation) y un refinamiento post-inferencia con **umbralización de Sauvola**, un DeepLabv3+ puede superar a modelos entrenados con 8x más datos. Detalle completo en `research/segmentacion-layout-one-shot-manuscritos-arabes.md` del vault de Obsidian.

**Antes de correr este notebook:**
1. En tu máquina local: `cd backend && python -m scripts.export_word_masks --out ../colab/dataset`
2. Comprime `colab/dataset/` en `dataset.zip` y súbelo aquí (celda de subida más abajo).

**Salida:** un archivo `word_segmenter.onnx` descargable, listo para usarse desde `segmentation_service.py` en el backend (con `onnxruntime`, sin depender de PyTorch en producción).

## 0. Setup

In [ ]:
!pip install -q scikit-image onnx onnxruntime
import torch, torchvision
print('torch', torch.__version__, '| cuda disponible:', torch.cuda.is_available())

## 1. Subir y descomprimir el dataset exportado

In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()  # selecciona dataset.zip
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('dataset')

images_dir = 'dataset/images'
masks_dir = 'dataset/masks'
ids = sorted(f.split('.')[0] for f in os.listdir(images_dir))
print(f'{len(ids)} páginas etiquetadas encontradas:', ids)
assert len(ids) >= 1, 'Exporta al menos 1 página con backend/scripts/export_word_masks.py'

## 2. Dataset + Generación Dinámica de Instancias

Reproduce el módulo del paper: redimensiona cada página a un tamaño fijo, la divide en un
grid de parches base sin solape, y añade `k` recortes aleatorios adicionales **por época**
(no fijos entre épocas) como augmentation — es la pieza que hace viable entrenar con 1-3
imágenes. Valores por defecto = los del paper (imagen 1344×2016, parches 672×672, k=12).

In [ ]:
import random
import numpy as np
from PIL import Image
from torch.utils.data import Dataset

RESIZE_TO = (2016, 1344)   # (alto, ancho) — igual que el paper; ajusta si tus folios son muy distintos en proporción
PATCH_SIZE = 672           # parches cuadrados PATCH_SIZE x PATCH_SIZE
RANDOM_CROPS_PER_EPOCH = 12

def load_pair(doc_id):
    img = Image.open(f'{images_dir}/{doc_id}.png').convert('RGB').resize((RESIZE_TO[1], RESIZE_TO[0]))
    mask = Image.open(f'{masks_dir}/{doc_id}.png').convert('L').resize((RESIZE_TO[1], RESIZE_TO[0]), Image.NEAREST)
    return np.array(img), (np.array(mask) > 127).astype(np.int64)

def grid_patches(img, mask, patch=PATCH_SIZE):
    h, w = mask.shape
    out = []
    for y in range(0, h - patch + 1, patch):
        for x in range(0, w - patch + 1, patch):
            out.append((img[y:y+patch, x:x+patch], mask[y:y+patch, x:x+patch]))
    return out

def random_crops(img, mask, n=RANDOM_CROPS_PER_EPOCH, patch=PATCH_SIZE):
    h, w = mask.shape
    out = []
    for _ in range(n):
        y = random.randint(0, h - patch)
        x = random.randint(0, w - patch)
        out.append((img[y:y+patch, x:x+patch], mask[y:y+patch, x:x+patch]))
    return out

class WordPatchDataset(Dataset):
    """Regenera los recortes aleatorios en cada época (Dynamic Instance Generation)."""
    def __init__(self, doc_ids):
        self.pairs = [load_pair(d) for d in doc_ids]
        self.regenerate()

    def regenerate(self):
        patches = []
        for img, mask in self.pairs:
            patches += grid_patches(img, mask)
            patches += random_crops(img, mask)
        self.patches = patches

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, i):
        img, mask = self.patches[i]
        img_t = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        mask_t = torch.from_numpy(mask).long()
        return img_t, mask_t

## 3. Split train/val

Con muy pocas páginas, usa la(s) misma(s) página(s) para train y valida contra recortes
distintos, o si tienes >=2 páginas, deja 1 fuera para validación real. Ajusta `val_ids`
manualmente según cuántas páginas hayas etiquetado.

In [ ]:
val_ids = ids[-1:] if len(ids) > 1 else ids   # última página como validación si hay >1
train_ids = [i for i in ids if i not in val_ids] or ids
print('train:', train_ids, '| val:', val_ids)

train_ds = WordPatchDataset(train_ids)
val_ds = WordPatchDataset(val_ids)
print(f'{len(train_ds)} parches de train, {len(val_ds)} parches de val (por época)')

## 4. Modelo: DeepLabv3+ (backbone ResNet50 preentrenado)

El paper usa Aligned XCeption; aquí usamos `deeplabv3_resnet50` de torchvision (pesos
preentrenados en COCO) por disponibilidad directa en Colab — el propio paper reporta que
el backbone no es la pieza crítica del método (sí lo son Dynamic Instance Generation +
Sauvola), así que este cambio es razonable. 2 clases: fondo / palabra.

In [ ]:
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = deeplabv3_resnet50(weights=DeepLabV3_ResNet50_Weights.DEFAULT)
model.classifier[4] = torch.nn.Conv2d(256, 2, kernel_size=1)          # fondo / palabra
model.aux_classifier[4] = torch.nn.Conv2d(256, 2, kernel_size=1)
model = model.to(device)
print('Modelo listo en', device)

## 5. Pérdida: Jaccard + Dice (igual que el paper)

In [ ]:
import torch.nn.functional as F

def jaccard_dice_loss(logits, target, eps=1e-6):
    probs = F.softmax(logits, dim=1)[:, 1]           # prob. de clase "palabra"
    target_f = target.float()
    inter = (probs * target_f).sum(dim=(1, 2))
    union = probs.sum(dim=(1, 2)) + target_f.sum(dim=(1, 2)) - inter
    jaccard = 1 - (inter + eps) / (union + eps)
    dice = 1 - (2 * inter + eps) / (probs.sum(dim=(1, 2)) + target_f.sum(dim=(1, 2)) + eps)
    return (jaccard.mean() + dice.mean()) / 2

## 6. Entrenamiento (Adam, early stopping — hiperparámetros del paper)

`lr=1e-3`, `weight_decay=1e-5`, máx. 200 épocas, corta si val no mejora en 20 épocas
consecutivas (con un margen/buffer de 50 épocas antes de poder cortar).

In [ ]:
from torch.utils.data import DataLoader

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
MAX_EPOCHS, PATIENCE, BUFFER = 200, 20, 50
best_val, epochs_no_improve, best_state = float('inf'), 0, None

for epoch in range(MAX_EPOCHS):
    train_ds.regenerate()  # nuevos recortes aleatorios cada época
    train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=4)

    model.train()
    train_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)['out']
        loss = jaccard_dice_loss(out, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    train_loss /= len(train_ds)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)['out']
            val_loss += jaccard_dice_loss(out, y).item() * x.size(0)
    val_loss /= len(val_ds)

    print(f'epoch {epoch+1:3d} | train {train_loss:.4f} | val {val_loss:.4f}')

    if val_loss < best_val:
        best_val, epochs_no_improve = val_loss, 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        epochs_no_improve += 1
        if epoch > BUFFER and epochs_no_improve >= PATIENCE:
            print(f'Early stopping en epoch {epoch+1} (sin mejora en {PATIENCE} épocas)')
            break

model.load_state_dict(best_state)
print('Mejor val loss:', best_val)

## 7. Refinamiento post-inferencia con Sauvola

Igual que el paper: `window_size=31`, `k=0.2` — calibra estos dos valores empíricamente
sobre tus propias páginas si los bordes de las cajas de palabra salen muy gruesos/delgados
(el propio paper señala esto como su principal limitación).

In [ ]:
from skimage.filters import threshold_sauvola
from skimage.color import rgb2gray

def predict_mask(model, img_rgb_uint8):
    x = torch.from_numpy(img_rgb_uint8).permute(2, 0, 1).float().unsqueeze(0) / 255.0
    with torch.no_grad():
        logits = model(x.to(device))['out']
        coarse = logits.argmax(dim=1).squeeze(0).cpu().numpy().astype(np.uint8)
    return coarse

def refine_with_sauvola(img_rgb_uint8, coarse_mask, window_size=31, k=0.2):
    gray = rgb2gray(img_rgb_uint8)
    thresh = threshold_sauvola(gray, window_size=window_size, k=k)
    ink = (gray < thresh)              # tinta = más oscuro que el umbral local
    return (coarse_mask.astype(bool) & ink).astype(np.uint8)  # intersección: refina bordes hacia la tinta real

## 8. Visualizar sobre una página de validación

In [ ]:
import matplotlib.pyplot as plt

sample_id = val_ids[0]
img, gt = load_pair(sample_id)
coarse = predict_mask(model, img)
refined = refine_with_sauvola(img, coarse)

fig, axes = plt.subplots(1, 4, figsize=(20, 6))
for ax, im, title in zip(axes, [img, gt, coarse, refined],
                          ['Original', 'GT (manual)', 'Predicción gruesa', 'Refinado (Sauvola)']):
    ax.imshow(im, cmap='gray' if im.ndim == 2 else None)
    ax.set_title(title)
    ax.axis('off')
plt.show()

## 9. Exportar a ONNX (para inferencia en el backend sin depender de PyTorch)

El backend actual (`segmentation_service.py`) solo usa OpenCV. Para no añadir PyTorch
como dependencia de producción, exporta a ONNX y usa `onnxruntime` (CPU) en su lugar —
mucho más liviano de instalar/mantener en el servidor.

In [ ]:
model.eval()
dummy = torch.randn(1, 3, PATCH_SIZE, PATCH_SIZE, device=device)
torch.onnx.export(
    model, dummy, 'word_segmenter.onnx',
    input_names=['image'], output_names=['logits'],
    dynamic_axes={'image': {0: 'batch', 2: 'height', 3: 'width'}, 'logits': {0: 'batch', 2: 'height', 3: 'width'}},
    opset_version=17,
)
files.download('word_segmenter.onnx')
print('Descarga word_segmenter.onnx y colócalo en backend/app/ml/word_segmenter.onnx')

## Próximo paso en el backend (no incluido en este notebook)

1. `pip install onnxruntime` en `backend/requirements.txt`.
2. Nueva función `segment_by_ml(image_path)` en `segmentation_service.py`: carga
   `word_segmenter.onnx` una vez al iniciar la app, corre inferencia por parches (mismo
   grid `PATCH_SIZE=672` usado aquí), aplica refinamiento Sauvola, convierte la máscara
   binaria en cajas con `cv2.findContours` — mismo formato de salida (`bbox_x/y/width/
   height`, `confidence`) que `segment_by_contour/projection/mser`, así encaja sin
   cambios en `run_segmentation()` ni en el resto de la app.
3. Registrar `"ml"` en el diccionario dispatcher de `run_segmentation()` y en el array
   `METHODS` de `frontend/src/components/labeling/LabelingPanel.tsx`.
4. Seguir el patrón *preview → apply* ya existente — nada más en la app necesita cambiar.